Vegetation Change in Aerial photographs of Urban Sprawl and foresty areas



In [1]:
%matplotlib inline

import cv2
import numpy as np 
from matplotlib import pyplot as plt 
from matplotlib import image as image
from pathlib import Path

In [2]:
#images are read from (parent)\Images, results go to (parent)\pairs
SRC_DIR = Path("..") / "Images"
OUT_DIR = Path("..") / "pairs"


#each item in folder looped through only looking for jpeg,jpg,png collecting file paths in a list. anything else is ignored. 
#turns them into lowercase and sorts them by name.
source_images = sorted(
    (p for p in SRC_DIR.iterdir() if p.suffix.lower() in {".jpeg", ".jpg", ".png"}),
    key=lambda p: p.name.casefold(),
)
print(f"Found {len(source_images)} images")#print how many images found

Found 10 images


In [ ]:
def find_divider(img, split_columns):
    #returns (start,end): the pixel range to remove between the two panels/images.
    #for a left/right image we work with columns.For a top/bottom image we
    #transpose it so rows become columns and the same code works for both.
    work = img if split_columns else img.transpose(1, 0, 2)
    h, w = work.shape[:2]
    #avoid other edges and divider is always near middle, search the middle 20% of the image (40% to 60% across)
    lo, hi = int(w * 0.4), int(w * 0.6)

    # case 1: white gap between images
    gray = cv2.cvtColor(work, cv2.COLOR_BGR2GRAY)#turn the image to black and white
    blur = cv2.GaussianBlur(gray, (5, 5), 0)#gaussian: blur it a bit so tiny details

    sobel = np.abs(cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3))#sobel: how much brightness changes left to right at every pixel
    sobel_score = np.median(sobel, axis=0)[lo:hi]#one number per column (middle 20% only), median so the edge has to run most of the way down

    canny = cv2.Canny(blur, 50, 150)#canny: marks only strong edges, pixel is 255 if its an edge and 0 if not
    canny_score = (canny > 0).mean(axis=0)[lo:hi]#one number per column: what fraction of the rows have an edge there

    # make both scores go from 0 to 1 so neither one counts more, then add them
    score = sobel_score / sobel_score.max() + canny_score / canny_score.max()

    # columns that score high are the edges, first and last one are the two sides of the gap
    strong = np.where(score >= 0.5 * score.max())[0] + lo#"+ lo" turns the position back into a column number of the full image
    start, end = int(strong[0]), int(strong[-1]) + 1

    # check the strip between the two edges is mostly white, if not its not a gap
    if (gray[:, start:end] > 240).mean() > 0.9:
        return start, end

    #case 2: no white gap, but the average colour is different on each side of the divider
    blur = cv2.GaussianBlur(work, (5, 5), 0)#smooth out images so less confusion
    
    best_x = w // 2 #fall back to the middle and consider best cut position
    best_score = 0 #how different the two sides are, higher is better
    for x in range(int(w * 0.46), int(w * 0.54)): #no gap means roughly a 50/50 split, so search an even narrower band (46% to 54%)
        left = blur[:, x - 8:x].mean(axis=1) # average colour of the 8 columns on the left, per row
        right = blur[:, x:x + 8].mean(axis=1) # average colour of the 8 columns on the right, per row
        difference = np.abs(right - left).sum(axis=1) #how different they are, per row
        score = np.median(difference)# typical difference down the whole height
        if score > best_score:
            best_score = score
            best_x = x

    return best_x, best_x

In [7]:
def split_image(source_path, pair_number):
    output_dir = OUT_DIR / f"{pair_number:02d}"
    output_dir.mkdir(parents=True, exist_ok=True)

    full_image = cv2.imread(str(source_path), cv2.IMREAD_COLOR)
    if full_image is None:
        raise ValueError(f"Could not read image: {source_path}")

    height, width = full_image.shape[:2]
    split_columns = width >= height

    start, end = find_divider(full_image, split_columns)

    if split_columns:
        before, after = full_image[:, :start], full_image[:, end:]
        orientation = "left/right"
    else:
        before, after = full_image[:start, :], full_image[end:, :]
        orientation = "top/bottom"

    for name, img in (("before", before), ("after", after)):
        if not cv2.imwrite(str(output_dir / f"{name}.png"), img):
            raise OSError(f"Could not write image: {output_dir / name}.png")

    return orientation, (start, end), before, after

In [ ]:
#preview: red lines show where find_divider thinks the cut is, check it by eye before saving anything
#one row per image, taller figure the more images there are
fig, axes = plt.subplots(len(source_images), 1, figsize=(12, 3.5 * len(source_images)))
axes = np.atleast_1d(axes)#with only one image matplotlib returns a single axis, this keeps it a list so the loop works

for ax, path in zip(axes, source_images):
    img = cv2.imread(str(path))
    split_columns = img.shape[1] >= img.shape[0]#wider than tall = side by side, otherwise one above the other
    start, end = find_divider(img, split_columns)

    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))#opencv is BGR, matplotlib needs RGB
    line = ax.axvline if split_columns else ax.axhline#vertical lines for left/right, horizontal for top/bottom
    #one line at each side of the divider, they land on the same spot when there is no gap
    line(start, color="red", lw=1)
    line(end, color="red", lw=1)
    ax.set_title(path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#the real split: saves before.png and after.png for every image into pairs/01 ... pairs/10
#start=1 so the folders are numbered 01 to 10 and not 00 to 09
for pair_number, source_path in enumerate(source_images, start=1):
    #split_image saves the two halves and returns what it did so we can print a summary
    orientation, (start, end), before, after = split_image(source_path, pair_number)
    #summary per image: cut direction, where the cut was, and size of each half (width x height)
    #shape[1] is the width and shape[0] is the height
    print(f"{pair_number:02d} {source_path.name}: {orientation}, cut at {start}-{end}, "
        f"before={before.shape[1]}x{before.shape[0]}, after={after.shape[1]}x{after.shape[0]}")